# KV Bench — Stratigraphic Strategy Benchmark (Modal)

**Goal**: Benchmark the Stratigraphic KV-cache strategy against baseline and other strategies on Llama-3.1-8B-Instruct.

**Run on**: Modal GPU (A100-80GB recommended)

---

## Cell 1: Environment Setup

In [ ]:
# === Modal Setup ===
%uv pip install -q flash-linear-attention transformers datasets accelerate matplotlib tqdm

import os
if not os.path.exists('PROJECT'):
    !git clone https://github.com/mohamedAtoui/KV-cache-optimization.git PROJECT
%cd PROJECT
!git pull origin main
%uv pip install -e .

# HuggingFace login (Llama-3.1-8B-Instruct is a gated model)
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    hf_token = input("Enter your HuggingFace token (https://huggingface.co/settings/tokens): ")
login(token=hf_token)

import torch
import logging

logging.basicConfig(level=logging.INFO)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.reset_peak_memory_stats()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## Cell 2: Quick Smoke Test (small run)

Run with `--max-samples 20 --max-seq-len 1024` to verify everything works before the full benchmark.

In [ ]:
!python -m kv_bench \
    --strategies baseline stratigraphic \
    --max-samples 20 \
    --max-seq-len 1024 \
    -v

## Cell 3: Full Benchmark — Baseline vs Stratigraphic

Full WikiText-2 evaluation. Settings are auto-detected based on GPU.

In [ ]:
!python -m kv_bench \
    --strategies baseline stratigraphic \
    --output results_stratigraphic.json \
    --markdown results_stratigraphic.md \
    -v

## Cell 4: Full Benchmark — All Strategies

Compare stratigraphic against all other strategies. Requires `--pattern-dir` for kv2state/hybrid.

In [ ]:
# Clone DuoAttention patterns (needed for kv2state and hybrid)
if not os.path.exists('duo-attention'):
    !git clone --depth 1 https://github.com/mit-han-lab/duo-attention.git duo-attention

PATTERN_DIR = 'duo-attention/attn_patterns/Meta-Llama-3.1-8B-Instruct'

!python -m kv_bench \
    --strategies baseline stratigraphic adaptive int8 int4 h2o snapkv kv2state hybrid \
    --pattern-dir {PATTERN_DIR} \
    --output results_all.json \
    --markdown results_all.md \
    -v

## Cell 5: Load & Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt

# Load whichever results file exists
results_file = 'results_all.json' if os.path.exists('results_all.json') else 'results_stratigraphic.json'
with open(results_file) as f:
    results = json.load(f)

names = [r['name'] for r in results]
ppls = [r['perplexity'] for r in results]
compressions = [r['compression_ratio'] for r in results]
mem_mb = [r['memory_kv_analytical_mb'] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Perplexity comparison
ax1 = axes[0]
colors = ['#2ecc71' if 'baseline' in n.lower() else '#e74c3c' if 'strat' in n.lower() else '#3498db' for n in names]
ax1.barh(names, ppls, color=colors)
ax1.set_xlabel('Perplexity')
ax1.set_title('Perplexity (lower = better)')
ax1.invert_yaxis()
for i, v in enumerate(ppls):
    ax1.text(v + 0.1, i, f'{v:.2f}', va='center')

# Plot 2: Compression ratio
ax2 = axes[1]
ax2.barh(names, compressions, color=colors)
ax2.set_xlabel('Compression Ratio')
ax2.set_title('Compression Ratio (higher = better)')
ax2.invert_yaxis()
for i, v in enumerate(compressions):
    ax2.text(v + 0.05, i, f'{v:.2f}x', va='center')

# Plot 3: Pareto frontier — PPL vs Compression
ax3 = axes[2]
for i, name in enumerate(names):
    marker = '*' if 'strat' in name.lower() else 'o'
    size = 200 if 'strat' in name.lower() else 80
    ax3.scatter(compressions[i], ppls[i], s=size, marker=marker, color=colors[i], zorder=5)
    ax3.annotate(name, (compressions[i], ppls[i]), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)
ax3.set_xlabel('Compression Ratio')
ax3.set_ylabel('Perplexity')
ax3.set_title('Quality vs Compression (bottom-right = ideal)')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kv_bench_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: kv_bench_results.png')

## Cell 6: Stratigraphic Per-Layer Analysis

Visualize how compression varies across layers (early layers compress more, late layers preserve more).

In [ ]:
from kv2state.stratigraphic import StratigraphicConfig
import numpy as np

config = StratigraphicConfig()
num_layers = 32
seq_len = 4096

# Compute per-layer zone fractions
layers = np.arange(num_layers)
fp16_fracs = [config.zone_surface * ((1 - config.lambda_) + config.lambda_ * l / (num_layers - 1)) for l in layers]
int8_fracs = [config.zone_shallow] * num_layers
int4_fracs = [config.zone_deep] * num_layers
evict_fracs = [1.0 - fp16 - config.zone_shallow - config.zone_deep for fp16 in fp16_fracs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Stacked bar: zone fractions per layer
ax1.bar(layers, fp16_fracs, label='FP16 (surface)', color='#2ecc71')
ax1.bar(layers, int8_fracs, bottom=fp16_fracs, label='INT8 (shallow)', color='#f39c12')
bottoms2 = [fp16 + int8 for fp16, int8 in zip(fp16_fracs, int8_fracs)]
ax1.bar(layers, int4_fracs, bottom=bottoms2, label='INT4 (deep)', color='#e74c3c')
bottoms3 = [b + int4 for b, int4 in zip(bottoms2, int4_fracs)]
ax1.bar(layers, evict_fracs, bottom=bottoms3, label='Evicted (bedrock)', color='#95a5a6')
ax1.set_xlabel('Layer Index')
ax1.set_ylabel('Token Fraction')
ax1.set_title('Stratigraphic Zone Distribution per Layer')
ax1.legend(loc='upper left')
ax1.set_ylim(0, 1.0)

# Line plot: FP16 fraction gradient
ax2.plot(layers, fp16_fracs, 'g-o', markersize=4, linewidth=2, label='FP16 fraction')
ax2.axhline(y=config.anchor_budget, color='red', linestyle='--', alpha=0.7, label=f'Anchor floor ({config.anchor_budget:.0%})')
ax2.axhline(y=config.zone_surface, color='blue', linestyle='--', alpha=0.7, label=f'Max surface ({config.zone_surface:.0%})')
ax2.fill_between(layers, config.anchor_budget, fp16_fracs, alpha=0.2, color='green')
ax2.set_xlabel('Layer Index')
ax2.set_ylabel('FP16 Fraction')
ax2.set_title(f'Inverse Layer Budget (λ={config.lambda_})')
ax2.legend()
ax2.set_ylim(0, 0.4)

plt.tight_layout()
plt.savefig('stratigraphic_layers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stratigraphic_layers.png')

# Print memory breakdown
from kv_bench.strategies.stratigraphic import StratigraphicStrategy
from kv_bench.strategies.baseline import FullKVBaseline
from types import SimpleNamespace

mock_config = SimpleNamespace(
    num_hidden_layers=32, num_attention_heads=32,
    num_key_value_heads=8, hidden_size=4096
)

strat = StratigraphicStrategy()
baseline = FullKVBaseline()

for sl in [1024, 2048, 4096, 8192]:
    full_mb = baseline.memory_bytes(sl, mock_config) / (1024**2)
    strat_mb = strat.memory_bytes(sl, mock_config) / (1024**2)
    ratio = full_mb / strat_mb
    print(f'seq_len={sl:5d}: Full KV={full_mb:7.1f} MB, Stratigraphic={strat_mb:7.1f} MB, Compression={ratio:.2f}x')

## Cell 7: Print Results Table

In [ ]:
# Print markdown results if available
for md_file in ['results_stratigraphic.md', 'results_all.md']:
    if os.path.exists(md_file):
        print(f'\n=== {md_file} ===')
        with open(md_file) as f:
            print(f.read())